# Forward Propagation, Backward Propagation, and Computational Graphs
:label:`sec_backprop`

So far, we have trained our models
with minibatch stochastic gradient descent.
However, when we implemented the algorithm,
we only worried about the calculations involved
in *forward propagation* through the model.
When it came time to calculate the gradients,
we just invoked the backpropagation function provided by the deep learning framework.

The automatic calculation of gradients
profoundly simplifies
the implementation of deep learning algorithms.
Before automatic differentiation,
even small changes to complicated models required
recalculating complicated derivatives by hand.
Surprisingly often, academic papers had to allocate
numerous pages to deriving update rules.
While we must continue to rely on automatic differentiation
so we can focus on the interesting parts,
you ought to know how these gradients
are calculated under the hood
if you want to go beyond a shallow
understanding of deep learning.

In this section, we take a deep dive
into the details of *backward propagation*
(more commonly called *backpropagation*).
To convey some insight for both the
techniques and their implementations,
we rely on some basic mathematics and computational graphs.
To start, we focus our exposition on
a one-hidden-layer MLP
with weight decay ($\ell_2$ regularization, to be described in subsequent chapters).

## Forward Propagation

*Forward propagation* (or *forward pass*) refers to the calculation and storage
of intermediate variables (including outputs)
for a neural network in order
from the input layer to the output layer.
We now work step-by-step through the mechanics
of a neural network with one hidden layer.
This may seem tedious but in the eternal words
of funk virtuoso James Brown,
you must "pay the cost to be the boss".


For the sake of simplicity, let's assume
that the input example is $\mathbf{x}\in \mathbb{R}^d$
and that our hidden layer does not include a bias term.
Here the intermediate variable is:

$$\mathbf{z}= \mathbf{W}^{(1)} \mathbf{x},$$

where $\mathbf{W}^{(1)} \in \mathbb{R}^{h \times d}$
is the weight parameter of the hidden layer.
After running the intermediate variable
$\mathbf{z}\in \mathbb{R}^h$ through the
activation function $\phi$
we obtain our hidden activation vector of length $h$:

$$\mathbf{h}= \phi (\mathbf{z}).$$

The hidden layer output $\mathbf{h}$
is also an intermediate variable.
Assuming that the parameters of the output layer
possess only a weight of
$\mathbf{W}^{(2)} \in \mathbb{R}^{q \times h}$,
we can obtain an output layer variable
with a vector of length $q$:

$$\mathbf{o}= \mathbf{W}^{(2)} \mathbf{h}.$$

Assuming that the loss function is $l$
and the example label is $y$,
we can then calculate the loss term
for a single data example,

$$L = l(\mathbf{o}, y).$$

As we will see the definition of $\ell_2$ regularization
to be introduced later,
given the hyperparameter $\lambda$,
the regularization term is

$$s = \frac{\lambda}{2} \left(\|\mathbf{W}^{(1)}\|_\textrm{F}^2 + \|\mathbf{W}^{(2)}\|_\textrm{F}^2\right),$$
:eqlabel:`eq_forward-s`

where the Frobenius norm of the matrix
is simply the $\ell_2$ norm applied
after flattening the matrix into a vector.
Finally, the model's regularized loss
on a given data example is:

$$J = L + s.$$

We refer to $J$ as the *objective function*
in the following discussion.


## Computational Graph of Forward Propagation

Plotting *computational graphs* helps us visualize
the dependencies of operators
and variables within the calculation.
:numref:`fig_forward` contains the graph associated
with the simple network described above,
where squares denote variables and circles denote operators.
The lower-left corner signifies the input
and the upper-right corner is the output.
Notice that the directions of the arrows
(which illustrate data flow)
are primarily rightward and upward.

![Computational graph of forward propagation.](../img/forward.svg)
:label:`fig_forward`

## Backpropagation

*Backpropagation* refers to the method of calculating
the gradient of neural network parameters.
In short, the method traverses the network in reverse order,
from the output to the input layer,
according to the *chain rule* from calculus.
The algorithm stores any intermediate variables
(partial derivatives)
required while calculating the gradient
with respect to some parameters.
Assume that we have functions
$\mathsf{Y}=f(\mathsf{X})$
and $\mathsf{Z}=g(\mathsf{Y})$,
in which the input and the output
$\mathsf{X}, \mathsf{Y}, \mathsf{Z}$
are tensors of arbitrary shapes.
By using the chain rule,
we can compute the derivative
of $\mathsf{Z}$ with respect to $\mathsf{X}$ via

$$\frac{\partial \mathsf{Z}}{\partial \mathsf{X}} = \textrm{prod}\left(\frac{\partial \mathsf{Z}}{\partial \mathsf{Y}}, \frac{\partial \mathsf{Y}}{\partial \mathsf{X}}\right).$$

Here we use the $\textrm{prod}$ operator
to multiply its arguments
after the necessary operations,
such as transposition and swapping input positions,
have been carried out.
For vectors, this is straightforward:
it is simply matrix--matrix multiplication.
For higher dimensional tensors,
we use the appropriate counterpart.
The operator $\textrm{prod}$ hides all the notational overhead.

Recall that
the parameters of the simple network with one hidden layer,
whose computational graph is in :numref:`fig_forward`,
are $\mathbf{W}^{(1)}$ and $\mathbf{W}^{(2)}$.
The objective of backpropagation is to
calculate the gradients $\partial J/\partial \mathbf{W}^{(1)}$
and $\partial J/\partial \mathbf{W}^{(2)}$.
To accomplish this, we apply the chain rule
and calculate, in turn, the gradient of
each intermediate variable and parameter.
The order of calculations are reversed
relative to those performed in forward propagation,
since we need to start with the outcome of the computational graph
and work our way towards the parameters.
The first step is to calculate the gradients
of the objective function $J=L+s$
with respect to the loss term $L$
and the regularization term $s$:

$$\frac{\partial J}{\partial L} = 1 \; \textrm{and} \; \frac{\partial J}{\partial s} = 1.$$

Next, we compute the gradient of the objective function
with respect to variable of the output layer $\mathbf{o}$
according to the chain rule:

$$
\frac{\partial J}{\partial \mathbf{o}}
= \textrm{prod}\left(\frac{\partial J}{\partial L}, \frac{\partial L}{\partial \mathbf{o}}\right)
= \frac{\partial L}{\partial \mathbf{o}}
\in \mathbb{R}^q.
$$

Next, we calculate the gradients
of the regularization term
with respect to both parameters:

$$\frac{\partial s}{\partial \mathbf{W}^{(1)}} = \lambda \mathbf{W}^{(1)}
\; \textrm{and} \;
\frac{\partial s}{\partial \mathbf{W}^{(2)}} = \lambda \mathbf{W}^{(2)}.$$

Now we are able to calculate the gradient
$\partial J/\partial \mathbf{W}^{(2)} \in \mathbb{R}^{q \times h}$
of the model parameters closest to the output layer.
Using the chain rule yields:

$$\frac{\partial J}{\partial \mathbf{W}^{(2)}}= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{o}}, \frac{\partial \mathbf{o}}{\partial \mathbf{W}^{(2)}}\right) + \textrm{prod}\left(\frac{\partial J}{\partial s}, \frac{\partial s}{\partial \mathbf{W}^{(2)}}\right)= \frac{\partial J}{\partial \mathbf{o}} \mathbf{h}^\top + \lambda \mathbf{W}^{(2)}.$$
:eqlabel:`eq_backprop-J-h`

To obtain the gradient with respect to $\mathbf{W}^{(1)}$
we need to continue backpropagation
along the output layer to the hidden layer.
The gradient with respect to the hidden layer output
$\partial J/\partial \mathbf{h} \in \mathbb{R}^h$ is given by


$$
\frac{\partial J}{\partial \mathbf{h}}
= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{o}}, \frac{\partial \mathbf{o}}{\partial \mathbf{h}}\right)
= {\mathbf{W}^{(2)}}^\top \frac{\partial J}{\partial \mathbf{o}}.
$$

Since the activation function $\phi$ applies elementwise,
calculating the gradient $\partial J/\partial \mathbf{z} \in \mathbb{R}^h$
of the intermediate variable $\mathbf{z}$
requires that we use the elementwise multiplication operator,
which we denote by $\odot$:

$$
\frac{\partial J}{\partial \mathbf{z}}
= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{h}}, \frac{\partial \mathbf{h}}{\partial \mathbf{z}}\right)
= \frac{\partial J}{\partial \mathbf{h}} \odot \phi'\left(\mathbf{z}\right).
$$

Finally, we can obtain the gradient
$\partial J/\partial \mathbf{W}^{(1)} \in \mathbb{R}^{h \times d}$
of the model parameters closest to the input layer.
According to the chain rule, we get

$$
\frac{\partial J}{\partial \mathbf{W}^{(1)}}
= \textrm{prod}\left(\frac{\partial J}{\partial \mathbf{z}}, \frac{\partial \mathbf{z}}{\partial \mathbf{W}^{(1)}}\right) + \textrm{prod}\left(\frac{\partial J}{\partial s}, \frac{\partial s}{\partial \mathbf{W}^{(1)}}\right)
= \frac{\partial J}{\partial \mathbf{z}} \mathbf{x}^\top + \lambda \mathbf{W}^{(1)}.
$$



## Training Neural Networks

When training neural networks,
forward and backward propagation depend on each other.
In particular, for forward propagation,
we traverse the computational graph in the direction of dependencies
and compute all the variables on its path.
These are then used for backpropagation
where the compute order on the graph is reversed.

Take the aforementioned simple network as an illustrative example.
On the one hand,
computing the regularization term :eqref:`eq_forward-s`
during forward propagation
depends on the current values of model parameters $\mathbf{W}^{(1)}$ and $\mathbf{W}^{(2)}$.
They are given by the optimization algorithm according to backpropagation in the most recent iteration.
On the other hand,
the gradient calculation for the parameter
:eqref:`eq_backprop-J-h` during backpropagation
depends on the current value of the hidden layer output $\mathbf{h}$,
which is given by forward propagation.


Therefore when training neural networks, once model parameters are initialized,
we alternate forward propagation with backpropagation,
updating model parameters using gradients given by backpropagation.
Note that backpropagation reuses the stored intermediate values from forward propagation to avoid duplicate calculations.
One of the consequences is that we need to retain
the intermediate values until backpropagation is complete.
This is also one of the reasons why training
requires significantly more memory than plain prediction.
Besides, the size of such intermediate values is roughly
proportional to the number of network layers and the batch size.
Thus,
training deeper networks using larger batch sizes
more easily leads to *out-of-memory* errors.


## Summary

Forward propagation sequentially calculates and stores intermediate variables within the computational graph defined by the neural network. It proceeds from the input to the output layer.
Backpropagation sequentially calculates and stores the gradients of intermediate variables and parameters within the neural network in the reversed order.
When training deep learning models, forward propagation and backpropagation are interdependent,
and training requires significantly more memory than prediction.


## Exercises

1. Assume that the inputs $\mathbf{X}$ to some scalar function $f$ are $n \times m$ matrices. What is the dimensionality of the gradient of $f$ with respect to $\mathbf{X}$?
1. Add a bias to the hidden layer of the model described in this section (you do not need to include bias in the regularization term).
    1. Draw the corresponding computational graph.
    1. Derive the forward and backward propagation equations.
1. Compute the memory footprint for training and prediction in the model described in this section.
1. Assume that you want to compute second derivatives. What happens to the computational graph? How long do you expect the calculation to take?
1. Assume that the computational graph is too large for your GPU.
    1. Can you partition it over more than one GPU?
    1. What are the advantages and disadvantages over training on a smaller minibatch?

[Discussions](https://discuss.d2l.ai/t/102)



1. Assume that the inputs $\mathbf{X}$ to some scalar function $f$ are $n \times m$ matrices. What is the dimensionality of the gradient of $f$ with respect to $\mathbf{X}$?


### 1. Dimensionality of the Gradient of a Scalar Function with Matrix Input

The gradient of a scalar function with respect to its input has the same dimensionality as the input itself. This is because the gradient represents the rate of change of the function with respect to each element of the input.

#### Mathematical Explanation:
If we have a scalar function $f: \mathbb{R}^{n \times m} \rightarrow \mathbb{R}$ that takes an $n \times m$ matrix $\mathbf{X}$ as input, then the gradient $\nabla_{\mathbf{X}} f$ is defined as:

$$\nabla_{\mathbf{X}} f = \begin{bmatrix} 
\frac{\partial f}{\partial X_{11}} & \frac{\partial f}{\partial X_{12}} & \cdots & \frac{\partial f}{\partial X_{1m}} \\
\frac{\partial f}{\partial X_{21}} & \frac{\partial f}{\partial X_{22}} & \cdots & \frac{\partial f}{\partial X_{2m}} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial f}{\partial X_{n1}} & \frac{\partial f}{\partial X_{n2}} & \cdots & \frac{\partial f}{\partial X_{nm}}
\end{bmatrix}$$

This is an $n \times m$ matrix, which is the same dimensionality as the input $\mathbf{X}$.

#### Intuitive Explanation:
Think of the gradient as a "sensitivity map" - for each element in the input matrix, the gradient tells you how sensitive the output is to small changes in that element. Since we need one sensitivity value for each input element, the gradient must have exactly the same shape as the input.

For example, if we're working with image data where $\mathbf{X}$ is a 28×28 pixel image (an $n=28, m=28$ matrix) and $f$ is a function that computes some scalar score from this image, then the gradient $\nabla_{\mathbf{X}} f$ will also be a 28×28 matrix. Each entry tells us how much a small change to that specific pixel would affect the final score.

#### In the Context of Neural Networks:
In backpropagation, we often need to compute gradients of scalar loss functions with respect to matrices of parameters (like weight matrices). The shape of these gradients matches the shape of the corresponding parameter matrices, which is why we can directly update the parameters using the gradients in gradient descent.

Therefore, the dimensionality of the gradient of $f$ with respect to $\mathbf{X}$ is $n \times m$.

2. Add a bias to the hidden layer of the model described in this section (you do not need to include bias in the regularization term).
    1. Draw the corresponding computational graph.
    1. Derive the forward and backward propagation equations.


### 2. Adding a Bias to the Hidden Layer

#### A. Computational Graph with Bias

When we add a bias term to the hidden layer, the computational graph changes slightly. Instead of just having $\mathbf{z} = \mathbf{W}^{(1)}\mathbf{x}$, we now have $\mathbf{z} = \mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$, where $\mathbf{b}^{(1)} \in \mathbb{R}^h$ is the bias vector for the hidden layer.

The modified computational graph would look like this (a text representation since we can't draw the actual graph here):

- Input $\mathbf{x}$ feeds into multiplication operation with $\mathbf{W}^{(1)}$
- The result $\mathbf{W}^{(1)}\mathbf{x}$ is added to bias $\mathbf{b}^{(1)}$ to produce $\mathbf{z}$
- $\mathbf{z}$ passes through activation function $\phi$ to produce $\mathbf{h}$
- $\mathbf{h}$ multiplies with $\mathbf{W}^{(2)}$ to produce $\mathbf{o}$
- $\mathbf{o}$ and label $y$ feed into loss function $l$ to produce $L$
- $\mathbf{W}^{(1)}$ and $\mathbf{W}^{(2)}$ feed into regularization term $s$
- $L$ and $s$ sum to produce objective function $J$

#### B. Forward and Backward Propagation Equations

**Forward Propagation Equations:**

1. First, compute the intermediate variable with the bias term:
   $$\mathbf{z} = \mathbf{W}^{(1)}\mathbf{x} + \mathbf{b}^{(1)}$$

2. Apply the activation function:
   $$\mathbf{h} = \phi(\mathbf{z})$$

3. Compute the output:
   $$\mathbf{o} = \mathbf{W}^{(2)}\mathbf{h}$$

4. Calculate the loss:
   $$L = l(\mathbf{o}, y)$$

5. Compute the regularization term (bias not included in regularization):
   $$s = \frac{\lambda}{2}(||\mathbf{W}^{(1)}||_F^2 + ||\mathbf{W}^{(2)}||_F^2)$$

6. Calculate the objective function:
   $$J = L + s$$

**Backward Propagation Equations:**

Starting from $\frac{\partial J}{\partial J} = 1$, we work backward through the computational graph:

1. Gradients with respect to the loss and regularization terms:
   $$\frac{\partial J}{\partial L} = 1 \quad \text{and} \quad \frac{\partial J}{\partial s} = 1$$

2. Gradient with respect to the output layer:
   $$\frac{\partial J}{\partial \mathbf{o}} = \frac{\partial L}{\partial \mathbf{o}} \in \mathbb{R}^q$$

3. Gradients of the regularization term with respect to weight parameters:
   $$\frac{\partial s}{\partial \mathbf{W}^{(1)}} = \lambda \mathbf{W}^{(1)} \quad \text{and} \quad \frac{\partial s}{\partial \mathbf{W}^{(2)}} = \lambda \mathbf{W}^{(2)}$$

4. Gradient with respect to $\mathbf{W}^{(2)}$:
   $$\frac{\partial J}{\partial \mathbf{W}^{(2)}} = \frac{\partial J}{\partial \mathbf{o}} \mathbf{h}^\top + \lambda \mathbf{W}^{(2)} \in \mathbb{R}^{q \times h}$$

5. Gradient with respect to the hidden layer output:
   $$\frac{\partial J}{\partial \mathbf{h}} = {\mathbf{W}^{(2)}}^\top \frac{\partial J}{\partial \mathbf{o}} \in \mathbb{R}^h$$

6. Gradient with respect to pre-activation output:
   $$\frac{\partial J}{\partial \mathbf{z}} = \frac{\partial J}{\partial \mathbf{h}} \odot \phi'(\mathbf{z}) \in \mathbb{R}^h$$

7. Gradient with respect to $\mathbf{W}^{(1)}$:
   $$\frac{\partial J}{\partial \mathbf{W}^{(1)}} = \frac{\partial J}{\partial \mathbf{z}} \mathbf{x}^\top + \lambda \mathbf{W}^{(1)} \in \mathbb{R}^{h \times d}$$

8. Gradient with respect to the bias (note that bias doesn't appear in regularization):
   $$\frac{\partial J}{\partial \mathbf{b}^{(1)}} = \frac{\partial J}{\partial \mathbf{z}} \in \mathbb{R}^h$$

The key difference from the original model is the addition of the bias term $\mathbf{b}^{(1)}$ in the forward pass and computing its gradient in the backward pass. Note that the bias gradient is simply equal to $\frac{\partial J}{\partial \mathbf{z}}$ because the bias connects directly to $\mathbf{z}$ with a weight of 1, and there is no regularization term for the bias. $\mathbf{z}$ with a weight of 1, and there is no regularization term for the bias.
```

3. Compute the memory footprint for training and prediction in the model described in this section.

### 3. Memory Footprint for Training and Prediction

To analyze the memory footprint, I'll break down all the variables we need to store during training and prediction, then count the total number of values stored.

#### Model Parameters

First, let's identify the model parameters that need to be stored for both training and prediction:
- Weight matrix for the hidden layer: $\mathbf{W}^{(1)} \in \mathbb{R}^{h \times d}$ - requires $h \times d$ values
- Weight matrix for the output layer: $\mathbf{W}^{(2)} \in \mathbb{R}^{q \times h}$ - requires $q \times h$ values

#### Training Memory Requirements

During training, we need to store:

1. **Model Parameters**:
  - $\mathbf{W}^{(1)}$: $h \times d$ values
  - $\mathbf{W}^{(2)}$: $q \times h$ values

2. **Intermediate Variables (for forward pass)**:
  - Input: $\mathbf{x} \in \mathbb{R}^d$ - requires $d$ values
  - Pre-activation hidden layer: $\mathbf{z} \in \mathbb{R}^h$ - requires $h$ values
  - Hidden layer activation: $\mathbf{h} \in \mathbb{R}^h$ - requires $h$ values
  - Output layer: $\mathbf{o} \in \mathbb{R}^q$ - requires $q$ values
  - Loss: $L$ - requires $1$ value
  - Regularization term: $s$ - requires $1$ value
  - Objective function: $J$ - requires $1$ value

3. **Gradients (for backward pass)**:
  - $\frac{\partial J}{\partial \mathbf{o}} \in \mathbb{R}^q$ - requires $q$ values
  - $\frac{\partial J}{\partial \mathbf{W}^{(2)}} \in \mathbb{R}^{q \times h}$ - requires $q \times h$ values
  - $\frac{\partial J}{\partial \mathbf{h}} \in \mathbb{R}^h$ - requires $h$ values
  - $\frac{\partial J}{\partial \mathbf{z}} \in \mathbb{R}^h$ - requires $h$ values
  - $\frac{\partial J}{\partial \mathbf{W}^{(1)}} \in \mathbb{R}^{h \times d}$ - requires $h \times d$ values

4. **Optimization Variables**:
  If using gradient descent with momentum or adaptive methods like Adam, we need additional memory for:
  - Parameter updates: same size as parameters ($h \times d + q \times h$)
  - Momentum/velocity terms: same size as parameters ($h \times d + q \times h$)

**Total Training Memory (for a single example):**
$h \times d + q \times h + d + h + h + q + 1 + 1 + 1 + q + q \times h + h + h + h \times d + 2(h \times d + q \times h)$

Simplifying:
$3(h \times d) + 3(q \times h) + d + 3h + 2q + 3$

For minibatch training with batch size $B$, the input and intermediate activations would be multiplied by $B$:
$3(h \times d) + 3(q \times h) + B \times (d + 3h + 2q) + 3$

#### Prediction Memory Requirements

During prediction (inference), we only need:

1. **Model Parameters**:
  - $\mathbf{W}^{(1)}$: $h \times d$ values
  - $\mathbf{W}^{(2)}$: $q \times h$ values

2. **Intermediate Variables** (for forward pass only):
  - Input: $\mathbf{x} \in \mathbb{R}^d$ - requires $d$ values
  - Pre-activation hidden layer: $\mathbf{z} \in \mathbb{R}^h$ - requires $h$ values
  - Hidden layer activation: $\mathbf{h} \in \mathbb{R}^h$ - requires $h$ values
  - Output layer: $\mathbf{o} \in \mathbb{R}^q$ - requires $q$ values

We don't need to store:
- Gradients
- Loss values
- Optimization variables

**Total Prediction Memory (for a single example):**
$h \times d + q \times h + d + h + h + q$

Simplifying:
$h \times d + q \times h + d + 2h + q$

For batch prediction with batch size $B$:
$h \times d + q \times h + B \times (d + 2h + q)$

#### Comparison

The ratio of training memory to prediction memory (for a single example) is approximately:
$\frac{3(h \times d) + 3(q \times h) + d + 3h + 2q + 3}{h \times d + q \times h + d + 2h + q} \approx 3$

This aligns with the text's statement that "training requires significantly more memory than plain prediction." The factor of approximately 3× comes from:
- Storing gradients (same size as parameters)
- Storing optimizer states (same size as parameters)
- Additional intermediate values needed for backpropagation

Note that this analysis assumes a simple optimizer. With more sophisticated optimizers (Adam, RMSProp), the memory requirements for training would be even larger due to additional optimizer states.

4. Assume that you want to compute second derivatives. What happens to the computational graph? How long do you expect the calculation to take?

### 4. Computing Second Derivatives: Impact on Computational Graph and Calculation Time

#### What Happens to the Computational Graph

When computing second derivatives, the computational graph undergoes a significant transformation:

1. **Graph Expansion**: The original computational graph must first be expanded to compute the first derivatives. Then, a new computational graph must be constructed to calculate derivatives of these first derivatives. This creates a "meta-graph" or a "graph of graphs."

2. **Nested Differentiation**: For a second derivative like $\frac{\partial^2 J}{\partial \mathbf{W}^{(1)2}}$, we first compute $\frac{\partial J}{\partial \mathbf{W}^{(1)}}$ through backpropagation, then treat this gradient itself as a function and differentiate it again.

3. **Additional Intermediate Variables**: We need to store all intermediate results from both the forward pass and the first backpropagation pass, as these become inputs to the second-order differentiation.

4. **Increased Interdependencies**: The second-order graph has far more connections, as each node in the first-order gradient computation can potentially depend on multiple nodes from the original graph.

5. **Hessian Matrix Structure**: For parameters with dimensions $h \times d$ and $q \times h$, the second derivatives form fourth-order tensors, which are extremely large and complex structures.

#### Expected Calculation Time

The computational complexity increases dramatically:

1. **Quadratic Growth**: If computing first-order gradients takes time $O(n)$ where $n$ is the number of parameters, then computing the full Hessian (all second derivatives) generally takes $O(n^2)$ time.

2. **Concrete Analysis**:
  - Forward pass: $O(hd + qh)$ operations
  - First backpropagation: Also $O(hd + qh)$ operations
  - Second backpropagation: $O((hd + qh)^2)$ operations

3. **For our specific model**:
  - The model has $(h \times d) + (q \times h)$ parameters
  - First derivatives require $O(h \times d + q \times h)$ calculations
  - Complete second derivatives (full Hessian) would require $O((h \times d + q \times h)^2)$ calculations

4. **Practical Implications**:
  - If first-order backpropagation takes time $T$, second-order backpropagation will take approximately $T \times (h \times d + q \times h)$ time
  - For even modestly sized networks (e.g., $h=100$, $d=784$, $q=10$), this becomes prohibitively expensive

5. **Memory Requirements**:
  - The full Hessian requires storing $(h \times d + q \times h)^2$ values
  - For the example dimensions above, this would be approximately $(78,400 + 1,000)^2 \approx 6.3 \times 10^9$ values

#### Practical Alternatives

Because of this computational burden, direct computation of the full Hessian is rarely done in practice. Instead:

1. **Hessian-vector products**: Can be computed more efficiently in $O(n)$ time without explicitly forming the Hessian
2. **Diagonal or block-diagonal approximations**: Capture the most important second-order information
3. **Quasi-Newton methods**: Approximate the Hessian using accumulated first-order gradient information (e.g., BFGS, L-BFGS)
4. **Gauss-Newton approximation**: Uses the Jacobian of the network outputs to approximate the Hessian of the loss

In summary, computing full second derivatives expands the computational graph dramatically, increasing both time and memory requirements by a factor proportional to the number of parameters. This makes direct computation of full second derivatives impractical for all but the smallest neural networks.

5. Assume that the computational graph is too large for your GPU.
    1. Can you partition it over more than one GPU?
    1. What are the advantages and disadvantages over training on a smaller minibatch?

### 5. Handling Large Computational Graphs with Multiple GPUs

#### A. Partitioning Computational Graphs Across Multiple GPUs

Yes, computational graphs that are too large for a single GPU can be partitioned across multiple GPUs using several strategies:

1. **Data Parallelism**:
  - Each GPU maintains a complete copy of the model
  - The training batch is split among GPUs
  - Each GPU computes gradients for its portion of data
  - Gradients are synchronized across GPUs (via all-reduce operations)
  - Parameters are updated synchronously
  - Implementation in frameworks: `torch.nn.DataParallel`, `torch.nn.parallel.DistributedDataParallel`

2. **Model Parallelism**:
  - The model itself is split across multiple GPUs
  - Different layers or components of the network reside on different GPUs
  - Forward and backward passes require communication between GPUs at layer boundaries
  - Useful when individual layers are too large for a single GPU's memory
  - Example: Placing different transformer blocks on different GPUs

3. **Pipeline Parallelism**:
  - Combines aspects of data and model parallelism
  - Model is divided into stages across GPUs
  - Different microbatches flow through the pipeline simultaneously
  - Reduces communication overhead compared to pure model parallelism
  - Examples: GPipe, PipeDream, Megatron-LM's pipeline implementation

4. **Tensor Parallelism**:
  - Individual operations (like matrix multiplications) are distributed across GPUs
  - Particularly useful for very large layers (e.g., attention heads in transformers)
  - Requires careful orchestration of communication between GPUs
  - Examples: Megatron-LM's tensor parallelism, DeepSpeed's ZeRO-3

5. **Hybrid Approaches**:
  - Combinations of the above strategies
  - Modern large-scale training often uses 3D parallelism (data + model + pipeline)
  - Frameworks like DeepSpeed and Megatron-LM support these hybrid approaches

#### B. Advantages and Disadvantages Compared to Smaller Minibatches

**Advantages of Multi-GPU Training vs. Smaller Minibatches:**

1. **Statistical Efficiency**:
  - Larger effective batch sizes provide more reliable gradient estimates
  - Can lead to better convergence behavior and potentially better final performance
  - Less noisy optimization trajectory

2. **Computational Efficiency**:
  - Higher hardware utilization, especially for matrix operations
  - Better amortization of communication overheads
  - Can reduce total training time significantly

3. **Model Capabilities**:
  - Enables training models that simply wouldn't fit on a single GPU
  - Allows exploration of larger architectures and longer sequences
  - Critical for scaling to state-of-the-art model sizes

4. **Throughput**:
  - More examples processed per second, leading to faster epoch completion
  - Can process the same dataset in less wall-clock time

**Disadvantages of Multi-GPU Training vs. Smaller Minibatches:**

1. **Implementation Complexity**:
  - Distributed training introduces significant complexity
  - Debugging becomes much harder
  - Requires careful handling of synchronization, communication, and failure recovery

2. **Communication Overhead**:
  - Network communication between GPUs can become a bottleneck
  - All-reduce operations scale with parameter count and GPU count
  - Can limit scaling efficiency as more GPUs are added

3. **Regularization Effects**:
  - Smaller batches provide a form of implicit regularization
  - Larger effective batches may require explicit regularization adjustments
  - Learning rate scaling becomes crucial for large batch training

4. **Hardware Costs**:
  - Obvious increase in hardware and energy costs
  - May require specialized hardware for inter-GPU communication
  - Higher infrastructure maintenance complexity

5. **Optimization Challenges**:
  - Large batch training can get stuck in sharper minima
  - May require careful learning rate scheduling and warmup
  - Potential degradation in generalization performance without proper tuning

6. **Diminishing Returns**:
  - Beyond certain batch sizes, statistical efficiency gains plateau
  - Linear scaling of computational resources doesn't always translate to linear speedups

In practice, the decision between multi-GPU training and smaller batches depends on specific needs, hardware availability, and the scale of the problem. Modern approaches often use multi-GPU training with techniques like gradient accumulation to balance these tradeoffs.